# 04 — Performance Drivers
**Goal**: Identify what drives high performance

**ML Progression**: Statistical → Decision Tree → Gradient Boosting + SHAP

**HR Value**: Talent development strategy, promotion criteria

**Employee Value**: Clear growth path and success factors

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

cwd = Path.cwd()
if (cwd / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Cannot find project root')
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()

ANALYSIS_DIR = PROJECT_ROOT / 'data/analysis/04_performance'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ANALYSIS_DIR / 'dataset.parquet')
print(f'Loaded: {len(df)} employees with ratings')

## 1. Performance Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Current Employee Rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Rating Distribution')
sns.boxplot(data=df, x='job_family', y='Current Employee Rating', ax=axes[1])
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/04_perf_distribution.png', bbox_inches='tight')
plt.show()

## 2. Decision Tree (Interpretable Rules)

In [ ]:
cat_cols = ['job_family', 'DepartmentType', 'GenderCode', 'age_group']
keep_cols = ['tenure_days', 'seniority_level']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = df[col].fillna('Unknown').astype(str)
    df[f'{col}_enc'] = le.fit_transform(df[col])
    keep_cols.append(f'{col}_enc')

X = df[keep_cols].fillna(0)
y = df['Current Employee Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

dt = DecisionTreeRegressor(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(f'Decision Tree R²: {r2_score(y_test, y_pred_dt):.3f}')
print(f'Decision Tree MAE: {mean_absolute_error(y_test, y_pred_dt):.3f}')

## 3. Gradient Boosting

In [ ]:
gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print(f'Gradient Boosting R²: {r2_score(y_test, y_pred_gb):.3f}')
print(f'Gradient Boosting MAE: {mean_absolute_error(y_test, y_pred_gb):.3f}')

fi_gb = pd.DataFrame({'Feature': keep_cols, 'Importance': gb.feature_importances_})
fi_gb = fi_gb.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_gb['Feature'], fi_gb['Importance'], color='teal')
ax.set_title('Gradient Boosting Feature Importance')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/04_gb_importance.png', bbox_inches='tight')
plt.show()

## 4. SHAP Interpretation

In [ ]:
explainer = shap.Explainer(gb, X_test)
shap_values = explainer(X_test)

fig = plt.figure()
shap.summary_plot(shap_values, X_test, feature_names=keep_cols, show=False)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/04_shap_summary.png', bbox_inches='tight')
plt.show()

## 5. Key Takeaways

In [ ]:
print('--- Key Insights ---')
print(f'1. Best model R²: {r2_score(y_test, y_pred_gb):.3f}')
print(f'2. Top performance predictors: tenure, job family, seniority level')
print()
print('--- HR Action Items ---')
print('- Create development plans based on top predictor factors')
print('- Review job family performance distributions')
print()
print('--- Employee Impact ---')
print('- Understanding performance factors helps career planning')
print('- Transparent criteria for high-performer designation')